# Kitty × ComfyUI on Google Colab

Runs ComfyUI on a Colab GPU and exposes it via a free cloudflared tunnel.

**One-time setup:** Upload your model files to Google Drive first (see cell 3).

**Each session:**
1. Run all cells top to bottom
2. Copy the `COMFY_URL=https://...` line printed in the last cell
3. Paste it into your local `.env` file
4. Restart the Kitty gateway: `pkill -f 'uvicorn' && python3.11 -m uvicorn gateway.app:app ...`
5. Generate images from the Image Gen panel in kitty-chat

In [ ]:
# Cell 1 — verify GPU
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print('GPU:', result.stdout.strip())
else:
    print('⚠ No GPU detected — go to Runtime → Change runtime type → GPU')

In [ ]:
# Cell 2 — clone ComfyUI and install dependencies
import os

if not os.path.exists('/content/ComfyUI'):
    !git clone https://github.com/comfyanonymous/ComfyUI /content/ComfyUI
else:
    print('ComfyUI already cloned, pulling latest...')
    !git -C /content/ComfyUI pull --ff-only

%cd /content/ComfyUI
!pip install -q -r requirements.txt
print('✓ ComfyUI ready')

In [ ]:
# Cell 3 — mount Google Drive and link your models
#
# Expected Drive layout (create these folders if missing):
#   MyDrive/AI/models/checkpoints/   ← homofidelis_v50.safetensors
#                                      photonicFusionSDXL_final.safetensors
#   MyDrive/AI/models/loras/         ← Muscle_Bear_Baker_v2_for_transfer.safetensors
#                                      erect_penis_epoch_80.safetensors
#
# Adjust DRIVE_BASE below if your layout differs.

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_BASE = '/content/drive/MyDrive/AI/models'
COMFY_BASE = '/content/ComfyUI/models'

import os

def link(src_dir: str, dst_dir: str):
    os.makedirs(dst_dir, exist_ok=True)
    if not os.path.exists(src_dir):
        print(f'  ⚠ Drive folder not found: {src_dir}')
        return
    for f in os.listdir(src_dir):
        src = os.path.join(src_dir, f)
        dst = os.path.join(dst_dir, f)
        if not os.path.exists(dst):
            os.symlink(src, dst)
            print(f'  linked {f}')
        else:
            print(f'  already linked {f}')

print('Linking checkpoints...')
link(f'{DRIVE_BASE}/checkpoints', f'{COMFY_BASE}/checkpoints')

print('Linking LoRAs...')
link(f'{DRIVE_BASE}/loras', f'{COMFY_BASE}/loras')

print('✓ Models linked')

In [ ]:
# Cell 4 — install cloudflared (free tunnel, no account needed)
import os
if not os.path.exists('/usr/local/bin/cloudflared'):
    !curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
        -o /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared
    print('✓ cloudflared installed')
else:
    print('✓ cloudflared already installed')

In [ ]:
# Cell 5 — start ComfyUI in the background
import subprocess, time, threading

comfy_proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '0.0.0.0', '--port', '8188',
     '--disable-auto-launch', '--preview-method', 'auto'],
    cwd='/content/ComfyUI',
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

def _tail_comfy():
    for line in comfy_proc.stdout:
        if any(kw in line for kw in ('Starting server', 'To see the GUI', 'VRAM', 'loaded')):
            print('[comfy]', line.rstrip())

threading.Thread(target=_tail_comfy, daemon=True).start()

# Wait until ComfyUI is ready
import urllib.request
for _ in range(60):
    try:
        urllib.request.urlopen('http://localhost:8188/system_stats', timeout=2)
        print('✓ ComfyUI is up')
        break
    except Exception:
        time.sleep(2)
else:
    print('⚠ ComfyUI did not start in time — check for errors above')

In [ ]:
# Cell 6 — open cloudflared tunnel and print the URL to paste into .env
import subprocess, threading, re, time

tunnel_url = None

tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8188'],
    stderr=subprocess.PIPE,
    stdout=subprocess.PIPE,
    text=True,
)

def _read_tunnel():
    global tunnel_url
    for line in tunnel_proc.stderr:
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m:
            tunnel_url = m.group()
            print(f'\n✓ Tunnel ready')
            print(f'\n  Add this line to your local .env and restart the gateway:')
            print(f'\n  COMFY_URL={tunnel_url}\n')

threading.Thread(target=_read_tunnel, daemon=True).start()

# Block until URL is captured (max 30s)
for _ in range(30):
    if tunnel_url:
        break
    time.sleep(1)
else:
    print('⚠ Tunnel URL not detected — check cloudflared output above')

In [ ]:
# Cell 7 (optional) — keep the runtime alive while you generate
# Run this cell and leave the tab open.  Interrupt the kernel to stop.
import time
print('Session alive. Interrupt kernel (■) to shut down ComfyUI.')
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print('Shutting down...')
    comfy_proc.terminate()
    tunnel_proc.terminate()